# Preparación de Datos para TP2 (Versión Final con los 3 Archivos)

### Objetivo
Este notebook consolida todo el proceso de carga, limpieza y fusión de datos. Se cargan los datos horarios, los datos de estaciones y las estadísticas históricas para generar un único DataFrame enriquecido y listo para el análisis de la Parte 2. Se conserva la columna `ESTACION` en el resultado final.

In [1]:
import pandas as pd
import numpy as np
import re

## 1. Carga de Datos

Se cargan los tres archivos de datos: horarios, de estaciones y de estadísticas normales, usando el método `read_fwf` con los parámetros que demostraron ser correctos para interpretar el formato de estos archivos.

In [2]:


print("--- Celda 1: Cargando los 3 archivos de datos ---")

try:
    # Archivo 1: datohorario20250715.txt (Ancho Fijo)
    df_hor = pd.read_fwf(
        "SMN_data/datohorario20250715.txt",
        encoding='latin1',
        skiprows=2,
        names=["FECHA", "HORA", "TEMP (°C)", "HUM (%)", "PNM (hPa)", "DD (g)", "FF (km/h)", "ESTACION"]
    )
   

    # Archivo 2: estaciones_smn.txt (Ancho Fijo)
    df_est = pd.read_fwf(
        'SMN_data/estaciones_smn.txt',
        encoding='latin1',
        skiprows=2,
        names=['ESTACION', 'PROVINCIA', 'LAT_GRAD', 'LAT_MIN', 'LON_GRAD', 'LON_MIN', 'ALTURA', 'NUM', 'NroOACI', 'LAT', 'LON']
    )
    
    # Archivo 3: Estadísticas...txt (Separado por Tabulaciones)
    df_stats = pd.read_csv(
        'SMN_data/Estadísticas normales Datos abiertos 1991-2020.txt',
        encoding='latin1',
        sep='\t',
        skiprows=7
    )
    df_stats.rename(columns={'Estación': 'ESTACION'}, inplace=True)
    
    print("✅ Carga de datos completada.")

except Exception as e:
    print(f"Error durante la carga de archivos: {e}")

--- Celda 1: Cargando los 3 archivos de datos ---
✅ Carga de datos completada.


In [3]:
# --- Comprobación de los DataFrames ---
print("\n--- 1. df_hor (Datos Horarios) ---")
display(df_hor.head())

print("\n--- 2. df_est (Datos de Estaciones) ---")
display(df_est.head())

print("\n--- 3. df_stats (Estadísticas Históricas) ---")
display(df_stats.head())


--- 1. df_hor (Datos Horarios) ---


,FECHA,HORA,TEMP (°C),HUM (%),PNM (hPa),DD (g),FF (km/h),ESTACION
0,15072025.0,0.0,13.7,88.0,1020.2,80.0,13.0,AEROPARQUE AERO
1,15072025.0,1.0,13.6,91.0,1019.6,80.0,11.0,AEROPARQUE AERO
2,15072025.0,2.0,13.4,91.0,1019.3,80.0,11.0,AEROPARQUE AERO
3,15072025.0,3.0,13.4,94.0,1018.7,80.0,13.0,AEROPARQUE AERO
4,15072025.0,4.0,13.7,91.0,1018.2,90.0,9.0,AEROPARQUE AERO



--- 2. df_est (Datos de Estaciones) ---


,ESTACION,PROVINCIA,LAT_GRAD,LAT_MIN,LON_GRAD,LON_MIN,ALTURA,NUM,NroOACI,LAT,LON
0,BASE BELGRANO II,ANTARTIDA,-77.0,52.0,-34.0,37.0,256.0,89034.0,SAYB,NaN,NaN
1,BASE CARLINI (EX JUBANY),ANTARTIDA,-62.0,14.0,-58.0,39.0,11.0,89053.0,SAYJ,NaN,NaN
2,BASE ESPERANZA,ANTARTIDA,-63.0,23.0,-56.0,59.0,24.0,88963.0,SAYE,NaN,NaN
3,BASE MARAMBIO,ANTARTIDA,-64.0,14.0,-56.0,37.0,198.0,89055.0,SAWB,NaN,NaN
4,BASE ORCADAS,ANTARTIDA,-60.0,44.0,-44.0,44.0,12.0,88968.0,SAYO,NaN,NaN



--- 3. df_stats (Estadísticas Históricas) ---


,ESTACION,Valor Medio de,Ene,Feb,Mar,Abr,May,Jun,Jul,Ago,Sep,Oct,Nov,Dic,Unnamed: 14
0,LA QUIACA OBSERVATORIO,Temperatura (°C),13.2,13.0,12.8,11.3,7.3,4.8,4.5,7.0,10.0,12.4,13.4,13.9,NaN
1,LA QUIACA OBSERVATORIO,Temperatura máxima (°C),20.6,20.4,20.6,20.3,17.8,16.3,16.1,18.0,20.0,21.7,22.5,22.2,NaN
2,LA QUIACA OBSERVATORIO,Temperatura mínima (°C),7.7,7.6,6.6,3.1,-2.5,-5.7,-6.2,-4.0,-0.4,3.3,5.5,7.3,NaN
3,LA QUIACA OBSERVATORIO,Humedad relativa (%),62.6,63.2,60.3,46.0,32.6,27.4,25.7,26.7,32.1,42.4,48.6,55.8,NaN
4,LA QUIACA OBSERVATORIO,Velocidad del Viento (km/h) (2011-2020),6.5,6.8,6.7,5.5,4.8,5.5,5.9,6.7,7.9,7.9,7.7,7.1,NaN


In [4]:
# Generamos las coordenadas de latitud y longitud a partir de los grados y minutos. Debemos contemplar  el signo de modo de obtener la conversión correcta.
df_est['LAT'] = (df_est['LAT_GRAD'] - df_est['LAT_MIN'] / 60)
df_est['LON'] = (df_est['LON_GRAD'] - df_est['LON_MIN'] / 60)

In [5]:
df_est.isna().mean().round(4)*100

ESTACION     0.00
PROVINCIA    1.67
LAT_GRAD     1.67
LAT_MIN      1.67
LON_GRAD     1.67
LON_MIN      1.67
ALTURA       1.67
NUM          1.67
NroOACI      2.50
LAT          1.67
LON          1.67
dtype: float64

In [6]:
# Unimos los DataFrames de datos horarios y estaciones por la columna 'ESTACION'
# Utilizamos un merge externo para conservar todas las estaciones, incluso si no tienen datos horarios
# --- esto último lo verificaremos más adelante ---
df_combinado = pd.merge(df_hor, df_est, on='ESTACION', how='outer')
df_combinado

,FECHA,HORA,TEMP (°C),HUM (%),PNM (hPa),DD (g),FF (km/h),ESTACION,PROVINCIA,LAT_GRAD,LAT_MIN,LON_GRAD,LON_MIN,ALTURA,NUM,NroOACI,LAT,LON
0,15072025.0,0.0,13.7,88.0,1020.2,80.0,13.0,AEROPARQUE AERO,CAPITAL FEDERAL,-34.0,33.0,-58.0,25.0,6.0,87582.0,SABE,-34.550000,-58.416667
1,15072025.0,1.0,13.6,91.0,1019.6,80.0,11.0,AEROPARQUE AERO,CAPITAL FEDERAL,-34.0,33.0,-58.0,25.0,6.0,87582.0,SABE,-34.550000,-58.416667
2,15072025.0,2.0,13.4,91.0,1019.3,80.0,11.0,AEROPARQUE AERO,CAPITAL FEDERAL,-34.0,33.0,-58.0,25.0,6.0,87582.0,SABE,-34.550000,-58.416667
3,15072025.0,3.0,13.4,94.0,1018.7,80.0,13.0,AEROPARQUE AERO,CAPITAL FEDERAL,-34.0,33.0,-58.0,25.0,6.0,87582.0,SABE,-34.550000,-58.416667
4,15072025.0,4.0,13.7,91.0,1018.2,90.0,9.0,AEROPARQUE AERO,CAPITAL FEDERAL,-34.0,33.0,-58.0,25.0,6.0,87582.0,SABE,-34.550000,-58.416667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2099,15072025.0,20.0,14.8,83.0,1015.8,180.0,19.0,VILLA REYNOLDS AE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2100,15072025.0,21.0,14.0,89.0,1016.4,180.0,17.0,VILLA REYNOLDS AE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2101,15072025.0,22.0,11.6,96.0,1017.9,140.0,41.0,VILLA REYNOLDS AE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2102,15072025.0,23.0,7.4,88.0,1021.3,180.0,19.0,VILLA REYNOLDS AE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# Obtenemos el dataframe que utilizaríamos para el modelado, eliminando las columnas que no aportan información o que presentan correlación muy alta con otras
df_ok = df_combinado.drop(columns=["FECHA", "ESTACION", "LAT_GRAD", "LAT_MIN", "LON_GRAD", "LON_MIN", "NUM", "NroOACI"])
df_ok

,HORA,TEMP (°C),HUM (%),PNM (hPa),DD (g),FF (km/h),PROVINCIA,ALTURA,LAT,LON
0,0.0,13.7,88.0,1020.2,80.0,13.0,CAPITAL FEDERAL,6.0,-34.550000,-58.416667
1,1.0,13.6,91.0,1019.6,80.0,11.0,CAPITAL FEDERAL,6.0,-34.550000,-58.416667
2,2.0,13.4,91.0,1019.3,80.0,11.0,CAPITAL FEDERAL,6.0,-34.550000,-58.416667
3,3.0,13.4,94.0,1018.7,80.0,13.0,CAPITAL FEDERAL,6.0,-34.550000,-58.416667
4,4.0,13.7,91.0,1018.2,90.0,9.0,CAPITAL FEDERAL,6.0,-34.550000,-58.416667
...,...,...,...,...,...,...,...,...,...,...
2099,20.0,14.8,83.0,1015.8,180.0,19.0,NaN,NaN,NaN,NaN
2100,21.0,14.0,89.0,1016.4,180.0,17.0,NaN,NaN,NaN,NaN
2101,22.0,11.6,96.0,1017.9,140.0,41.0,NaN,NaN,NaN,NaN
2102,23.0,7.4,88.0,1021.3,180.0,19.0,NaN,NaN,NaN,NaN


In [9]:
import pandas as pd
import re

# --- 1. Definir la función de limpieza ---
# Es crucial para que las claves de los 3 archivos coincidan
def limpiar_nombre_estacion(nombre):
    nombre = str(nombre).upper().replace("AERO", "").replace("OBS", "").replace("B.A.", "").strip()
    nombre = re.sub(r'\(.*?\)', '', nombre).strip()
    nombre = re.sub(r'[^\w\s]', '', nombre)
    nombre = re.sub(r'\s+', ' ', nombre).strip()
    return nombre

# --- 2. Crear 'df_merged' (unión de datos horarios y estaciones) ---
print("--- Celda de Fusión Completa ---")

# a) Crear la clave de unión en df_hor y df_est
df_hor['ESTACION_KEY'] = df_hor['ESTACION'].apply(limpiar_nombre_estacion)
df_est['ESTACION_KEY'] = df_est['ESTACION'].apply(limpiar_nombre_estacion)

# b) Crear la columna FECHA_HORA para el análisis temporal
# ======================== INICIO DE LA CORRECCIÓN ========================
# Se eliminan filas con fechas nulas para evitar errores en la conversión.
# Esto hace que el código sea más robusto y evita el TypeError si la celda se ejecuta varias veces.
df_hor.dropna(subset=['FECHA'], inplace=True)

# Se ajusta el formato a '%d%m%Y' y se convierte a integer.
df_hor['FECHA'] = pd.to_datetime(df_hor['FECHA'].astype(int), format='%d%m%Y', errors='coerce')
# ========================= FIN DE LA CORRECCIÓN ==========================

df_hor['HORA'] = pd.to_timedelta(df_hor['HORA'], unit='h')
df_hor['FECHA_HORA'] = df_hor['FECHA'] + df_hor['HORA']

# c) Realizar la primera fusión para crear df_merged
df_merged = pd.merge(df_hor, df_est, on='ESTACION_KEY', how='left')

print("✅ df_merged creado exitosamente.")

# --- 3. Procesar df_stats y fusionar con df_merged ---

# a) Crear la misma clave de unión en df_stats
df_stats['ESTACION_KEY'] = df_stats['ESTACION'].apply(limpiar_nombre_estacion)

# b) Filtrar y transformar los datos de temperatura histórica
df_stats_temp = df_stats[df_stats['Valor Medio de'] == 'Temperatura (°C)'].copy()
df_stats_long = df_stats_temp.melt(
    id_vars=['ESTACION_KEY'],
    value_vars=['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic'],
    var_name='MES_NOMBRE',
    value_name='TEMP_MEDIA_HIST'
)
meses_map = {'Ene': 1, 'Feb': 2, 'Mar': 3, 'Abr': 4, 'May': 5, 'Jun': 6, 'Jul': 7, 'Ago': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dic': 12}
df_stats_long['MES'] = df_stats_long['MES_NOMBRE'].map(meses_map)

# c) Preparar df_merged para la fusión final
df_merged['MES'] = df_merged['FECHA_HORA'].dt.month

# d) Realizar la fusión final
df_final = pd.merge(
    df_merged,
    df_stats_long[['ESTACION_KEY', 'MES', 'TEMP_MEDIA_HIST']],
    on=['ESTACION_KEY', 'MES'],
    how='left'
)

# --- 4. Verificación ---
print("\n✅ Fusión final con datos históricos completada.")
print(f"Forma del DataFrame final: {df_final.shape}")

print("\nPrimeras filas del resultado final (df_final):")
display(df_final.head())

# Guardar el resultado
df_final.to_csv('SMN_data/datos_meteorologicos_enriquecidos.csv', index=False)
print("\n✅ DataFrame final guardado como 'datos_meteorologicos_enriquecidos.csv'")


--- Celda de Fusión Completa ---
✅ df_merged creado exitosamente.

✅ Fusión final con datos históricos completada.
Forma del DataFrame final: (2035, 23)

Primeras filas del resultado final (df_final):


,FECHA,HORA,TEMP (°C),HUM (%),PNM (hPa),DD (g),FF (km/h),ESTACION_x,ESTACION_KEY,FECHA_HORA,...,LAT_MIN,LON_GRAD,LON_MIN,ALTURA,NUM,NroOACI,LAT,LON,MES,TEMP_MEDIA_HIST
0,2025-07-15,0 days 00:00:00,13.7,88.0,1020.2,80.0,13.0,AEROPARQUE AERO,PARQUE,2025-07-15 00:00:00,...,33.0,-58.0,25.0,6.0,87582.0,SABE,-34.55,-58.416667,7,11.3
1,2025-07-15,0 days 01:00:00,13.6,91.0,1019.6,80.0,11.0,AEROPARQUE AERO,PARQUE,2025-07-15 01:00:00,...,33.0,-58.0,25.0,6.0,87582.0,SABE,-34.55,-58.416667,7,11.3
2,2025-07-15,0 days 02:00:00,13.4,91.0,1019.3,80.0,11.0,AEROPARQUE AERO,PARQUE,2025-07-15 02:00:00,...,33.0,-58.0,25.0,6.0,87582.0,SABE,-34.55,-58.416667,7,11.3
3,2025-07-15,0 days 03:00:00,13.4,94.0,1018.7,80.0,13.0,AEROPARQUE AERO,PARQUE,2025-07-15 03:00:00,...,33.0,-58.0,25.0,6.0,87582.0,SABE,-34.55,-58.416667,7,11.3
4,2025-07-15,0 days 04:00:00,13.7,91.0,1018.2,90.0,9.0,AEROPARQUE AERO,PARQUE,2025-07-15 04:00:00,...,33.0,-58.0,25.0,6.0,87582.0,SABE,-34.55,-58.416667,7,11.3



✅ DataFrame final guardado como 'datos_meteorologicos_enriquecidos.csv'
